## Scan Date Match Diagnostic
Verify scan-date matching by comparing the scan Excel (T1_01) against the
combined CSV using visit code mapping (M0→Baseline, M12→FU1, …, M60→FU5).

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

BASE = Path('/dss/dsshome1/0A/di54lup/DELCODE/STRATIFICATION')
DATA = BASE / 'data'
INTERMEDIATE = DATA / 'intermediate'
SCAN_XLSX = DATA / 'study_metadata' / 'restingstate_scan_dates_M0_M60.xlsx'
COMBINED_CSV = INTERMEDIATE / 'combined' / 'all_visits.csv'

VISIT_MAP = {
    'M0': 'BASELINE', 'M12': 'Follow-up 1', 'M24': 'Follow-up 2',
    'M36': 'Follow-up 3', 'M48': 'Follow-up 4', 'M60': 'Follow-up 5',
}

# ── Load data ──────────────────────────────────────────────────────────
scans = pd.read_excel(SCAN_XLSX, sheet_name='T1_01')
scans = scans[['pseudonym', 'visit', 'scan_date']].dropna(subset=['scan_date']).copy()
scans['scan_date_dt'] = pd.to_datetime(scans['scan_date'], errors='coerce')
scans = scans.dropna(subset=['scan_date_dt'])
scans['mapped_visnam'] = scans['visit'].map(VISIT_MAP)

comb = pd.read_csv(COMBINED_CSV)
comb['_merge_key'] = comb['visnam'].fillna('BASELINE')
comb_patients = set(comb['Repseudonym'])
scan_patients = set(scans['pseudonym'])

# ── Build key sets ─────────────────────────────────────────────────────
scan_keys = set(zip(scans['pseudonym'], scans['mapped_visnam']))
comb_keys = set(zip(comb['Repseudonym'], comb['_merge_key']))
matched_keys = scan_keys & comb_keys
unused_scan_keys = scan_keys - comb_keys
unmatched_comb_keys = comb_keys - scan_keys

# ── Summary ───────────────────────────────────────────────────────────
print('=' * 70)
print('SCAN DATE MATCH DIAGNOSTIC (visit code mapping)')
print('=' * 70)
print(f'Scans (T1_01): {len(scans)} rows, {len(scan_patients)} patients')
print(f'Combined CSV:  {len(comb)} rows, {len(comb_patients)} patients')
print(f'Shared patients: {len(scan_patients & comb_patients)}')
print(f'\n✓ Matched (patient + visit): {len(matched_keys)}')

# ── Unused scans ──────────────────────────────────────────────────────
unused_df = scans[scans.apply(lambda r: (r['pseudonym'], r['mapped_visnam']) in unused_scan_keys, axis=1)].copy()
unused_df['reason'] = unused_df['pseudonym'].apply(
    lambda p: 'Patient not in combined CSV (filtered as relative)' if p not in comb_patients
    else 'Patient exists but this visit is missing in combined CSV')

print(f'\n✗ Unused scans: {len(unused_df)}')
for reason, grp in unused_df.groupby('reason'):
    print(f'  • {reason}: {len(grp)}')

# Show the ones where patient exists
patient_exists = unused_df[unused_df['pseudonym'].isin(comb_patients)]
if len(patient_exists) > 0:
    print(f'\n  Detail — scans where patient exists but visit missing:')
    display(patient_exists[['pseudonym', 'visit', 'scan_date_dt']].rename(
        columns={'scan_date_dt': 'scan_date'}))

# ── Unmatched combined rows ────────────────────────────────────────────
unmatched_comb = comb[comb.apply(lambda r: (r['Repseudonym'], r['_merge_key']) in unmatched_comb_keys, axis=1)].copy()
# Classify reason
has_scan_visits = {'BASELINE', 'Follow-up 1', 'Follow-up 2', 'Follow-up 3', 'Follow-up 4', 'Follow-up 5'}
unmatched_comb['reason'] = unmatched_comb.apply(lambda r:
    'Tel visit (no scan expected)' if 'Tel' in str(r['_merge_key'])
    else 'Follow-up 6+ (beyond M60, no scan data)' if r['_merge_key'] not in has_scan_visits
    else 'Patient not in scan data' if r['Repseudonym'] not in scan_patients
    else 'Patient has scan data but not for this visit', axis=1)

print(f'\n✗ Combined CSV rows without scan: {len(unmatched_comb)}')
for reason, grp in unmatched_comb.groupby('reason'):
    print(f'  • {reason}: {len(grp)}')


SCAN DATE MATCH DIAGNOSTIC (visit code mapping)
Scans (T1_01): 3321 rows, 964 patients
Combined CSV:  4605 rows, 866 patients
Shared patients: 857

✓ Matched (patient + visit): 2943

✗ Unused scans: 378
  • Patient exists but this visit is missing in combined CSV: 11
  • Patient not in combined CSV (filtered as relative): 367

  Detail — scans where patient exists but visit missing:


,pseudonym,visit,scan_date
461,2275a05f4,M48,2019-10-07
522,27044a887,M48,2021-10-26
590,2bf7ca999,M48,2022-06-07
683,31cc7f41f,M48,2020-02-28
739,3737eef10,M48,2019-10-10
835,3f8021b71,M48,2021-11-02
887,4240c00eb,M48,2022-01-12
1797,8796c61b4,M48,2021-07-28
2002,964ab82c5,M48,2021-07-28
2751,d366bf38a,M48,2019-09-16



✗ Combined CSV rows without scan: 1662
  • Follow-up 6+ (beyond M60, no scan data): 749
  • Patient has scan data but not for this visit: 779
  • Patient not in scan data: 36
  • Tel visit (no scan expected): 98


In [2]:
import pandas as pd
from pathlib import Path

BASE = Path('/dss/dsshome1/0A/di54lup/DELCODE/STRATIFICATION')
INTERMEDIATE = BASE / 'data' / 'intermediate'

cohorts = {
    'Healthy':           INTERMEDIATE / 'healthy'              / 'all_visits.csv',
    'MCI Converter':     INTERMEDIATE / 'mci' / 'converter'    / 'all_visits.csv',
    'MCI Non-Converter': INTERMEDIATE / 'mci' / 'non_converter'/ 'all_visits.csv',
    'AD':                INTERMEDIATE / 'ad'                   / 'cognitive_all_visits.csv',
}

dfs = {}
rows = []
for name, path in cohorts.items():
    df = pd.read_csv(path)
    dfs[name] = df
    rows.append({'Cohort': name, 'Total Rows': len(df), 'Unique Patients': df['Repseudonym'].nunique()})

summary = pd.DataFrame(rows)
total = pd.DataFrame([{'Cohort': 'TOTAL', 'Total Rows': summary['Total Rows'].sum(),
                       'Unique Patients': summary['Unique Patients'].sum()}])
summary = pd.concat([summary, total], ignore_index=True)

print('Cohort Summary')
print('=' * 55)
display(summary.style.hide(axis='index'))


Cohort Summary


Cohort,Total Rows,Unique Patients
Healthy,964,176
MCI Converter,191,39
MCI Non-Converter,2113,451
AD,236,108
TOTAL,3504,774


In [3]:
# ── Overlap check ────────────────────────────────────────────────────
names = list(dfs.keys())
overlaps = []
for i in range(len(names)):
    for j in range(i + 1, len(names)):
        a = set(dfs[names[i]]['Repseudonym'])
        b = set(dfs[names[j]]['Repseudonym'])
        common = a & b
        if common:
            overlaps.append({'Cohort A': names[i], 'Cohort B': names[j], 'Shared Patients': len(common)})

if overlaps:
    print('Patient Overlaps Between Cohorts')
    print('=' * 55)
    display(pd.DataFrame(overlaps).style.hide(axis='index'))
else:
    print('✓ No patient overlaps between any cohorts')

Patient Overlaps Between Cohorts


Cohort A,Cohort B,Shared Patients
MCI Converter,AD,39


In [4]:
# ── Copy cohort CSVs to export/ directory ─────────────────────────────
import shutil

export_dir = BASE / 'data' / 'export'
export_dir.mkdir(parents=True, exist_ok=True)

export_map = {
    INTERMEDIATE / 'healthy'               / 'all_visits.csv': 'healthy_all_visits.csv',
    INTERMEDIATE / 'mci' / 'converter'     / 'all_visits.csv': 'mci_converter_all_visits.csv',
    INTERMEDIATE / 'mci' / 'non_converter' / 'all_visits.csv': 'mci_non_converter_all_visits.csv',
    INTERMEDIATE / 'ad'                    / 'cognitive_all_visits.csv': 'ad_all_visits.csv',
}

for src, dst_name in export_map.items():
    dst = export_dir / dst_name
    shutil.copy2(src, dst)
    print(f'✓ {src.relative_to(BASE)} → export/{dst_name}')

print(f'\n✓ All cohorts exported to {export_dir}')


✓ data/intermediate/healthy/all_visits.csv → export/healthy_all_visits.csv
✓ data/intermediate/mci/converter/all_visits.csv → export/mci_converter_all_visits.csv
✓ data/intermediate/mci/non_converter/all_visits.csv → export/mci_non_converter_all_visits.csv
✓ data/intermediate/ad/cognitive_all_visits.csv → export/ad_all_visits.csv

✓ All cohorts exported to /dss/dsshome1/0A/di54lup/DELCODE/STRATIFICATION/data/export


## AD Without Prior MCI (Non-Baseline)
Identify patients who:
1. Develop AD (strict or cognitive-only) at a **follow-up** visit (not Baseline)
2. Have **no prior MCI** visit at all

Then extract **all visits** for these patients into a highlighted Excel.

In [5]:
# ── AD patients without prior MCI, AD only at follow-up ───────────────
mci_fv = pd.read_csv(INTERMEDIATE / 'mci' / 'first_visit.csv')
mci_patients = set(mci_fv['Repseudonym'])

# cognitive_first_visit includes ALL AD patients (strict is a subset)
ad_cog_fv  = pd.read_csv(INTERMEDIATE / 'ad' / 'cognitive_first_visit.csv')
ad_strict_fv = pd.read_csv(INTERMEDIATE / 'ad' / 'strict_first_visit.csv')

# Union of all AD patients (cognitive already includes strict)
all_ad_ids = set(ad_cog_fv['Repseudonym'])

# Step 1: AD patients with no prior MCI
ad_no_mci = all_ad_ids - mci_patients

# Step 2: Among those, keep only patients whose first AD visit is NOT Baseline
# Check both cognitive and strict first visits
cog_bl = ad_cog_fv[ad_cog_fv['Repseudonym'].isin(ad_no_mci) & (ad_cog_fv['file'] == 'BASELINE')]
strict_bl = ad_strict_fv[ad_strict_fv['Repseudonym'].isin(ad_no_mci) & (ad_strict_fv['file'] == 'BASELINE')]
has_ad_at_baseline = set(cog_bl['Repseudonym']) | set(strict_bl['Repseudonym'])

# Final set: no MCI, no AD at Baseline (neither strict nor cognitive)
ad_no_mci_no_bl_ids = ad_no_mci - has_ad_at_baseline

print(f'Total AD patients (cognitive + strict): {len(all_ad_ids)}')
print(f'AD patients with no prior MCI: {len(ad_no_mci)}')
print(f'  - with AD (strict or cognitive) at Baseline: {len(has_ad_at_baseline)}')
print(f'  - with AD only at follow-up (NOT Baseline): {len(ad_no_mci_no_bl_ids)}')
print()

# Show their IDs
df_show = ad_cog_fv[ad_cog_fv['Repseudonym'].isin(ad_no_mci_no_bl_ids)].sort_values('Repseudonym')
print('IDs:')
for _, r in df_show.iterrows():
    print(f"  {r['Repseudonym']}  (file={r['file']}, visdate={r['visdate']}, "
          f"mmstot={r['mmstot']}, cdrglobal={r['cdrglobal']})")


Total AD patients (cognitive + strict): 108
AD patients with no prior MCI: 69
  - with AD (strict or cognitive) at Baseline: 42
  - with AD only at follow-up (NOT Baseline): 27

IDs:
  01dc83c85  (file=FOLLOWUP, visdate=20-09-2016, mmstot=16.0, cdrglobal=1.0)
  07e48dffa  (file=FOLLOWUP, visdate=24-08-2020, mmstot=23.0, cdrglobal=1.0)
  1230b7af2  (file=FOLLOWUP, visdate=29-10-2018, mmstot=21.0, cdrglobal=1.0)
  1e36df350  (file=FOLLOWUP, visdate=25-01-2016, mmstot=14.0, cdrglobal=1.0)
  2c30e6116  (file=FOLLOWUP, visdate=29-05-2018, mmstot=22.0, cdrglobal=1.0)
  40c6470f2  (file=FOLLOWUP, visdate=20-07-2015, mmstot=15.0, cdrglobal=1.0)
  428978e36  (file=FOLLOWUP, visdate=17-12-2018, mmstot=23.0, cdrglobal=1.0)
  434050aec  (file=FOLLOWUP, visdate=06-09-2018, mmstot=21.0, cdrglobal=2.0)
  58f3f8bfb  (file=FOLLOWUP, visdate=27-07-2020, mmstot=20.0, cdrglobal=1.0)
  60a8dd7f5  (file=FOLLOWUP, visdate=29-07-2019, mmstot=17.0, cdrglobal=1.0)
  632000a93  (file=FOLLOWUP, visdate=21-11-2019

### Why are these patients' earlier visits not classified as MCI?
Check each non-AD visit against the three MCI criteria and show why it fails.

In [ ]:
# ── Check why pre-AD visits don't qualify as MCI ──────────────────────
full_df = pd.read_csv(INTERMEDIATE / 'combined' / 'all_visits.csv')
for col in ['mmstot', 'cdrglobal', 'ratio_Abeta42_40', 'ratio_Abeta42_phosphotau181']:
    full_df[col] = pd.to_numeric(full_df[col], errors='coerce')

mci_all = pd.read_csv(INTERMEDIATE / 'mci' / 'all_visits.csv')
mci_visit_keys = set(zip(mci_all['Repseudonym'], mci_all['visdate']))

print('=== Per-patient trajectory analysis ===')
print('MCI Criteria reminder:')
print('  C1: mmstot 26-30 AND cdrglobal 0.5-1')
print('  C2: mmstot > 26 AND cdrglobal < 0.5 AND abnormal biomarkers')
print('  C3: mmstot 24-28 AND cdrglobal = 0.5 AND normal biomarkers')
print()

reasons_summary = {}

for pid in sorted(ad_no_mci_no_bl_ids):
    pv = full_df[full_df['Repseudonym'] == pid].copy()
    pv['_vd'] = pd.to_datetime(pv['visdate'], dayfirst=True, errors='coerce')
    pv = pv.sort_values('_vd')
    print(f'--- {pid} ({len(pv)} visits) ---')
    for _, r in pv.iterrows():
        mmse = r['mmstot']
        cdr  = r['cdrglobal']
        r40  = r['ratio_Abeta42_40']
        rpt  = r['ratio_Abeta42_phosphotau181']

        # Check each MCI criterion
        c1 = (pd.notna(mmse) and pd.notna(cdr) and
              26 <= mmse <= 30 and 0.5 <= cdr <= 1)
        c2 = (pd.notna(mmse) and pd.notna(cdr) and
              mmse > 26 and cdr < 0.5 and
              pd.notna(r40) and r40 <= 0.08 and
              pd.notna(rpt) and rpt < 9.68)
        c3 = (pd.notna(mmse) and pd.notna(cdr) and
              24 <= mmse <= 28 and cdr == 0.5 and
              pd.notna(r40) and r40 > 0.08 and
              pd.notna(rpt) and rpt >= 9.68)
        ad_met = (pd.notna(mmse) and mmse <= 24 and
                  pd.notna(cdr) and cdr >= 1)

        tags = []
        if c1: tags.append('MCI-C1')
        if c2: tags.append('MCI-C2')
        if c3: tags.append('MCI-C3')
        if ad_met: tags.append('AD')
        label = ' + '.join(tags) if tags else 'NONE'

        # Reason why NOT MCI (for non-AD visits)
        reason = ''
        if not c1 and not c2 and not c3 and not ad_met:
            if pd.isna(mmse) or pd.isna(cdr):
                reason = 'missing mmstot/cdrglobal'
            elif mmse < 24:
                reason = f'mmse={mmse:.0f} too low for any MCI criterion (<24)'
            elif mmse <= 25 and cdr >= 0.5:
                reason = (f'mmse={mmse:.0f} too low for C1 (26-30); '
                          f'C3 needs 24-28 + cdr=0.5 + normal biomarkers')
                if cdr == 0.5:
                    if pd.isna(r40) or pd.isna(rpt):
                        reason += ' -> biomarkers MISSING'
                    elif r40 <= 0.08 or rpt < 9.68:
                        reason += ' -> biomarkers ABNORMAL (not normal)'
            elif mmse >= 26 and cdr < 0.5:
                reason = 'cognitively normal; C2 needs abnormal biomarkers'
                if pd.isna(r40) or pd.isna(rpt):
                    reason += ' -> biomarkers MISSING'
                elif r40 > 0.08 or rpt >= 9.68:
                    reason += ' -> biomarkers NORMAL'
            else:
                reason = f'mmse={mmse}, cdr={cdr}'

        r40_str = f'{r40:.4f}' if pd.notna(r40) else 'NaN'
        rpt_str = f'{rpt:.2f}' if pd.notna(rpt) else 'NaN'
        print(f'  {r["visdate"]:>11} {r["file"]:>8}  '
              f'mmse={str(mmse):>4} cdr={str(cdr):>3}  '
              f'r40={r40_str:>7} rpt={rpt_str:>6}  '
              f'-> {label}  {reason}')

        if not ad_met and reason:
            key = reason.split(';')[0].split(' ->')[0].strip()
            reasons_summary[key] = reasons_summary.get(key, 0) + 1
    print()

print('=== SUMMARY: Why pre-AD visits are not MCI ===')
for reason, count in sorted(reasons_summary.items(), key=lambda x: -x[1]):
    print(f'  {count:3d}x  {reason}')


## Export AD Without Prior MCI — Highlighted Excel
Extract **all visits** for AD patients who have no prior MCI visit,
with the same highlighting as the AD `highlighted.xlsx`:
- **Dark red**: strict AD visits (cognitive + biomarkers)
- **Light red**: cognitive-only AD visits
- **No color**: non-AD visits for these patients

In [6]:
from openpyxl import Workbook
from openpyxl.styles import PatternFill, Font

# ── Extract ALL visits for these patients from full dataset ────────────
full = pd.read_csv(INTERMEDIATE / 'combined' / 'all_visits.csv')
df_subset = full[full['Repseudonym'].isin(ad_no_mci_no_bl_ids)].copy()

# Load AD visit datasets for highlighting
ad_strict = pd.read_csv(INTERMEDIATE / 'ad' / 'strict_all_visits.csv')
ad_cog = pd.read_csv(INTERMEDIATE / 'ad' / 'cognitive_all_visits.csv')

strict_keys = set(zip(ad_strict['Repseudonym'], ad_strict['visdate']))
cog_keys = set(zip(ad_cog['Repseudonym'], ad_cog['visdate'])) - strict_keys

# Classify rows
dark_red_idx = set()
light_red_idx = set()
for idx, row in df_subset.iterrows():
    key = (row['Repseudonym'], row['visdate'])
    if key in strict_keys:
        dark_red_idx.add(idx)
    elif key in cog_keys:
        light_red_idx.add(idx)

print(f'AD patients (no MCI, no AD at Baseline): {len(ad_no_mci_no_bl_ids)}')
print(f'Total visits: {len(df_subset)}')
print(f'  Dark red (strict AD): {len(dark_red_idx)}')
print(f'  Light red (cognitive-only AD): {len(light_red_idx)}')
print(f'  Uncolored: {len(df_subset) - len(dark_red_idx) - len(light_red_idx)}')

# ── Write highlighted Excel ────────────────────────────────────────────
output_xlsx = INTERMEDIATE / 'ad_no_previous_mci.xlsx'

wb = Workbook()
ws = wb.active
ws.title = 'AD No Prior MCI'

display_cols = ['file', 'Repseudonym', 'visdate', 'scan_date', 'scan_year',
                'prmdiag', 'mmstot', 'cdrglobal', 'ratio_Abeta42_40',
                'ratio_Abeta42_phosphotau181']
display_cols = [c for c in display_cols if c in df_subset.columns]

dark_red_fill = PatternFill(start_color='ff0606', end_color='ff0606', fill_type='solid')
light_red_fill = PatternFill(start_color='ff8080', end_color='ff8080', fill_type='solid')
white_font = Font(color='FFFFFF')
header_fill = PatternFill(start_color='333333', end_color='333333', fill_type='solid')
header_font = Font(color='FFFFFF', bold=True)

# Header row
for col_idx, col_name in enumerate(display_cols, 1):
    cell = ws.cell(row=1, column=col_idx, value=col_name)
    cell.fill = header_fill
    cell.font = header_font

# Data rows
for excel_row, (df_idx, row_data) in enumerate(df_subset.iterrows(), 2):
    is_dark = df_idx in dark_red_idx
    is_light = df_idx in light_red_idx
    for col_idx, col_name in enumerate(display_cols, 1):
        value = row_data[col_name]
        if pd.isna(value):
            value = ''
        cell = ws.cell(row=excel_row, column=col_idx, value=str(value))
        if is_dark:
            cell.fill = dark_red_fill
            cell.font = white_font
        elif is_light:
            cell.fill = light_red_fill

# Auto-adjust column widths
for col_idx, col_name in enumerate(display_cols, 1):
    max_len = len(str(col_name))
    for row in ws.iter_rows(min_row=2, max_row=min(100, ws.max_row),
                            min_col=col_idx, max_col=col_idx):
        for cell in row:
            if cell.value:
                max_len = max(max_len, len(str(cell.value)))
    ws.column_dimensions[ws.cell(row=1, column=col_idx).column_letter].width = min(max_len + 2, 40)

wb.save(output_xlsx)
print(f'\n\u2713 Saved to: {output_xlsx}')
print(f'  File size: {output_xlsx.stat().st_size:,} bytes')


AD patients (no MCI, no AD at Baseline): 27
Total visits: 113
  Dark red (strict AD): 5
  Light red (cognitive-only AD): 49
  Uncolored: 59

✓ Saved to: /dss/dsshome1/0A/di54lup/DELCODE/STRATIFICATION/data/intermediate/ad_no_previous_mci.xlsx
  File size: 11,122 bytes
